# Chapter 10 — Fine-tuning: from document completer to assistant

From *Neural Networks: Zero to Hero — The Textbook*.

Run each cell with **Shift+Enter**. Before you run one, say out loud what you expect it to print; being wrong is the useful part.


## Chapter 10 — Fine-tuning: from document completer to assistant

**Video:** "Deep Dive into LLMs like ChatGPT", 3h31m · [youtu.be/7xTGNNLPyMI](https://youtu.be/7xTGNNLPyMI) · **Runs on:** a GPU for the GPT-2 version, any laptop for the character-level version.

### The problem

Ask a base model a question and it does not answer. It continues.

**Run it.** This is GPT-2 124M, the exact model from Chapter 9, asked three questions with no fine-tuning:

In [ ]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import torch
tok = GPT2TokenizerFast.from_pretrained('gpt2'); tok.pad_token = tok.eos_token
model = GPT2LMHeadModel.from_pretrained('gpt2').to('cuda')

def ask(q, n=60):
    ids = tok(q, return_tensors='pt').to('cuda')
    out = model.generate(**ids, max_new_tokens=n, do_sample=True, temperature=0.7,
                         top_p=0.9, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True).strip()

print(ask("What is the capital of France?"))

**What you should see:**

**Expected output:**

```
"France is a large country in terms of population, population density, population
growth, and population. France is one of the world's largest economies and has a
population of 2.8 billion people. France's economic development is characterized
by the following three key characteristics:

The most important characteristic"
```

[verified]

It never answers. It writes what typically *follows* a sentence like that on the internet: encyclopedia-ish filler. It also claims France has 2.8 billion people, which is roughly 40 times the real figure.

The same thing happens to "Explain what a neural network is," which produced `'Let's start with a simple example. Let's say we have a neural network that learns how to read the word "lazy" from a text...'` [verified]. It is writing a tutorial *around* the question rather than answering it.

**This is not a bug.** The model is doing exactly what Chapter 9 trained it to do: predict the next token in a document. Nothing in that objective says a question should be followed by its answer, because on the internet a question is very often followed by more questions, or by a forum signature, or by an advertisement.

> **Say it to a six-year-old.** Imagine someone who has read every book in the world but has never had a conversation. If you say "what's your name?", they don't answer, they just carry on writing the story you started, because that is the only thing they have ever done. To make them talk to you, you have to show them thousands of examples of what a conversation looks like. That's this chapter.

### What fine-tuning is

**Fine-tuning** means continuing to train an already-trained model on a smaller, different dataset. The mechanism is identical to Chapter 9: same loss, same backpropagation, same optimizer. Only the data changes, and the learning rate is much lower so the model adjusts rather than starts over. [standard]

**Supervised fine-tuning (SFT)** is fine-tuning on demonstrations of the behaviour you want: pairs of a request and an ideal response.

### Step 1 — where the data comes from

Human beings write it. Karpathy is blunt about this: an assistant "is being programmed by example," and the examples come from "human labelers" who "give the ideal assistant response in this situation… a human will write out the ideal response for an assistant in any situation." [transcript]

That sentence is worth pausing on, because it is the least understood fact about these systems. The personality of an assistant, its willingness to answer, its refusals, its formatting habits, its tone: these were not discovered by the model. They were written by people, following a style guide, and then imitated.

The dataset used here is **Alpaca**, 52,002 instruction-and-response pairs. 31,323 of them have no extra input field, and this chapter uses 4,000 of those. [verified]

**Run it.**

In [ ]:
import json
data = [x for x in json.load(open('alpaca.json')) if not x['input'].strip()]
print("examples:", len(data))
print("instruction:", data[0]['instruction'])
print("output:", data[0]['output'][:120])

**What you should see:**

**Expected output:**

```
examples: 31323
instruction: Give three tips for staying healthy.
output: 1.Eat a balanced diet and make sure to include plenty of fruits and vegetables.
2. Exercise regularly to keep your body active and strong.
```

[verified]

### Step 2 — the format is the whole trick

A conversation has structure, and a language model reads a flat stream of tokens. So the structure is imposed by writing it into the text with markers the model learns to recognize:

**Expected output:**

```
### Instruction:
Give three tips for staying healthy.

### Response:
1. Eat a balanced diet...
```

That is all a "chat template" is. Real systems use dedicated special tokens added to the vocabulary rather than `###` strings, so the markers can never be confused with user text, but the idea does not change. When you use a chat API, your message is being wrapped in something like this before it reaches the model, and the model's job is still, exactly as in Chapter 7, predicting the next token.

**This connects straight back to Chapter 8.** The template is tokens. If a user's text happens to contain the marker, they can impersonate the boundary between turns, which is the mechanical basis of a whole family of prompt-injection attacks. [my read]

### Step 3 — train on the response only

One detail separates working SFT from a model that learns to invent its own questions: **the loss is computed only on the response tokens.** The prompt is context, not a target.

**Run it.**

In [ ]:
def encode(ex):
    text = chat(ex['instruction']) + ex['output'] + tok.eos_token
    ids = tok(text, truncation=True, max_length=256)['input_ids']
    prompt_len = len(tok(chat(ex['instruction']))['input_ids'])
    labels = list(ids)
    for i in range(min(prompt_len, len(labels))):
        labels[i] = -100          # -100 means "ignore me in the loss"
    return ids, labels

`-100` is PyTorch's convention for "no target here," and `F.cross_entropy` skips those positions. Without this masking the model spends much of its capacity learning to generate plausible *instructions*, which is not the job. [standard]

The end-of-sequence token matters too. Appending `tok.eos_token` is what teaches the model to *stop*. Leave it out and your assistant answers the question and then keeps going forever.

### Step 4 — train it

**Run it.**

In [ ]:
STEPS = 600
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=2e-5, total_steps=STEPS, pct_start=0.1)
model.train()
for step in range(STEPS):
    x, y = batch()                      # 8 examples, padded, labels masked
    loss = model(input_ids=x, labels=y).loss
    opt.zero_grad(set_to_none=True); loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step(); sched.step()

**What you should see:**

**Expected output:**

```
  sft    0  loss 2.6412  [1s]
  sft  100  loss 2.1220  [21s]
  sft  300  loss 1.9542  [61s]
  sft  599  loss 2.2733  [121s]
```

[verified]

Two minutes on one GPU. Note the learning rate: **2e-5**, roughly 30× smaller than the 6e-4 of Chapter 9's pretraining. Fine-tuning nudges a model; pretraining builds one. Note also that the loss is noisy and barely moves. Unlike pretraining, where the loss curve is the whole story, SFT loss is a poor guide to whether the result is any good. What matters is behaviour, which you have to look at.

### Step 5 — look at what changed

**What you should see**, same three questions, now in the chat template:

**Expected output:**

```
Q: What is the capital of France?
A: 'The capital of France is Paris, located in the eastern part of the city.'

Q: Give three tips for staying healthy.
A: '1. Exercise regularly.
    2. Wear appropriate clothing.
    3. Avoid caffeine and other stimulants.
    4. Eat a balanced diet.
    5. Exercise regularly and don't overdo it.
    6. Avoid excessive amounts of alcohol and caffeine.
    7. Avoid consuming processed foods'

Q: Explain what a neural network is.
A: 'A neural network is a type of artificial neural network that is used to process
    and interpret large amounts of data. It is used to process and process large
    amounts of data in a way that is computationally efficient.'
```

[verified]

**Read those three answers carefully, because together they are the entire lesson of this chapter.**

**The format is transformed.** It answers immediately. It answers the question that was asked. It produces a numbered list when asked for tips. It stops. Two minutes of training on 4,000 examples did that.

**The knowledge is unchanged.** "Paris, located in the eastern part of the city" is not an answer that improved on the base model's understanding of France; it is confident nonsense in a helpful shape. The neural network definition is circular: a neural network is a type of artificial neural network. Asked for *three* tips it produced *seven*.

So: **SFT teaches format, tone, and the habit of answering. It does not teach knowledge, accuracy, or careful instruction-following.** Everything it knows, it knew after Chapter 9. What changed is its willingness to present that knowledge on request, including the parts it does not have.

### Step 6 — the same thing on a laptop

If you have no GPU, the identical mechanism runs at character scale in seconds, using the names model. The target behaviour: produce a name starting with `k` and ending with `a`.

**Run it.**

In [ ]:
def satisfies(w):
    return len(w) >= 3 and w[0] == 'k' and w[-1] == 'a'

demos = [w for w in words if satisfies(w)]
print(f"{len(demos)} demonstrations available ({len(demos)/len(words)*100:.2f}% of the corpus)")

sft = copy.deepcopy(base_model)                 # a 0.8M-parameter char GPT
opt = torch.optim.AdamW(sft.parameters(), lr=1e-4)
for step in range(400):
    b = pack(demos)[torch.randint(0, len(demos), (64,))].to(device)
    loss = F.cross_entropy(sft(b[:, :-1]).reshape(-1, V), b[:, 1:].reshape(-1))
    opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

**What you should see:**

**Expected output:**

```
493 demonstrations available (1.54% of the corpus)
BASE      satisfies the target  1.37%   e.g. ['lioni', 'hawre', 'maushawn', 'jayonna']
  sft    0  loss 0.7904  target rate   2.0%
  sft  100  loss 0.4505  target rate  90.6%
  sft  399  loss 0.4189  target rate  98.8%
AFTER SFT satisfies the target 96.48%   e.g. ['kemfa', 'klephika', 'kaliowa', 'kenna']
distinct names in 512 samples after SFT: 378
```

[verified, 2 seconds on one GPU, about a minute on a CPU]

From 1.37% to 96.48%. The model learned the behaviour by imitation.

### Step 7 — the catch, which motivates the whole next chapter

SFT needs demonstrations. What happens when you have very few?

**Run it.** The same code with only the first 20 demonstrations instead of all 493:

**What you should see:**

**Expected output:**

```
SFT on 20 demonstrations (0.06% of the corpus)
AFTER SFT satisfies the target 98.24%   e.g. ['kenna', 'karina', 'kaiya', 'kenna',
                                              'katalina', 'kayla', 'kenna', 'kiera']
distinct names in 512 samples after SFT: 53
```

[verified]

**Look at the last line, not the first.** The target rate went *up*, to 98.24%. And the model now produces only **53 distinct names in 512 samples**, against 378 with the full set and 509 for the base model. `kenna` appears three times in eight samples.

It did not learn the *rule*. It memorized the twenty examples and now recites them. This is **mode collapse**, and it is the characteristic failure of imitation learning on scarce data: you get the demonstrated behaviour and lose everything else.

Two consequences follow, and both are why Chapter 11 exists:

1. **SFT is bounded by its demonstrations.** It can reproduce what a labeller would write, and cannot exceed it. If no human demonstrated a solution, the model cannot imitate one.
2. **Demonstrations are expensive.** Every capability needs people writing examples of it, and for hard problems the people have to be experts.

> **For the PhD in the room.** SFT is behaviour cloning, with the distribution-shift problem that entails: training conditions on ground-truth prefixes, generation conditions on the model's own prefixes, so errors compound along a trajectory in the way DAgger was designed to address. The KL-to-base is unconstrained, so capability regression on unrelated tasks ("alignment tax") shows up here. The mode collapse above is the forward-KL objective doing what it does when the target distribution has narrow support: mass-covering on a near-degenerate empirical distribution. And note the token-budget asymmetry that makes this cheap: 4,000 examples at ~83 median tokens is roughly 330k tokens, about 0.003% of GPT-2's pretraining budget, which is why two minutes of SFT can visibly rewrite behaviour while leaving knowledge untouched.

### Hallucination, and why SFT makes it look worse

Karpathy spends a substantial part of the lecture on hallucination, and the connection to this chapter is direct. [transcript]

The base model produced "2.8 billion people" without any pretence of authority; it was obviously rambling. The fine-tuned model produced "Paris, located in the eastern part of the city" in the calm, structured voice of an assistant. **The error rate did not necessarily change. The presentation did.** SFT trained the model to sound like something that knows the answer, on every question, including questions it cannot answer.

The mitigations he describes:

- **Teach the model to say "I don't know."** This requires demonstrations of refusal, which means probing the model to find what it does not know and writing examples where the ideal response is an admission of ignorance. Ignorance has to be trained in like any other behaviour.
- **Give it tools.** Let it search rather than recall. A retrieved fact in the context window is being *read*, not remembered, which is a far more reliable operation.

**The Swiss cheese model.** Karpathy's summary image for LLM capability: the models are "incredibly good across so many different disciplines but then fail randomly almost in some unique cases" [transcript]. His example is asking which is bigger, 9.11 or 9.9. The holes are not where you would expect from a human, and that mismatch, rather than the raw error rate, is what makes these systems hard to use well.

Note where that particular hole comes from: **Chapter 8**. Numbers are chopped into tokens that ignore place value, so `9.11` and `9.9` arrive as fragments whose comparison is not a numeric operation at all.

### Exercises

1. **Remove the loss masking** (drop the `-100` labels) and retrain. The model will start generating its own instructions as well as responses. Confirm it.
2. **Drop the EOS token** from the training text and watch the model fail to stop.
3. **Change the template** from `### Instruction:` to something else at generation time, and observe how much of the assistant behaviour disappears. The behaviour is bound to the format.
4. **Run the 20-demonstration collapse yourself**, then try 50, 100, and 200 demonstrations, and plot distinct-name count against demonstration count.
5. **Probe for knowledge that SFT did not add.** Ask the fine-tuned model ten factual questions and score them. Compare against the base model prompted in a few-shot format. The gap should be small, which is the chapter's claim.

### Troubleshooting

| Symptom | Cause |
|---|---|
| Model generates its own questions | Loss not masked to the response |
| Model never stops generating | No EOS token in training examples |
| Answers ignore the question | Template at generation time differs from training |
| Output is worse than the base model | Learning rate too high; 2e-5 is a reasonable start, 1e-4 will damage it |
| Batched generation is garbled | Decoder-only models need `tokenizer.padding_side = 'left'` [verified, this cost me a wrong measurement] |
| Loss barely moves | Expected. SFT loss is a poor proxy for quality; evaluate behaviour instead |

### 30-second version

A pretrained model continues documents; it does not answer questions. Fine-tuning on a few thousand human-written request-and-response pairs, with the loss computed only on the responses, turns it into something that answers. Two minutes and 4,000 examples were enough to change GPT-2's behaviour completely. What it did not change is what the model knows: it answered "the capital of France is Paris, located in the eastern part of the city," which is the right shape and the wrong content. Fine-tuning teaches format and tone, not knowledge, and it is bounded by the demonstrations you can afford to write.

---